# Evaluation: final scores on the test split

This notebook scores every model on the held-out test split, once. The protocol:

1. Build each model with the settings stored in `models.BEST_PARAMS`, which were chosen on the validation split in `03_models.ipynb`. The majority baseline has no settings.
2. Fit on the training split only. The validation split is not added back, so the tuned settings stay matched to the amount of data they were tuned on.
3. Predict class probabilities for the test split one time and keep them in `test_proba`. Every later section reads those probabilities. Nothing is refitted or rescored after this point, and no setting changes after the test numbers are seen.
4. Report average AUC and macro-F1 from `src/metrics.py`, each with a 95% bootstrap interval.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

import pandas as pd  # noqa: E402

from src import data, metrics, models  # noqa: E402

In [ ]:
df = data.load()
train, valid, test = data.split(df)
train_p, valid_p, test_p = data.preprocess(train, valid, test)  # fills use train statistics only
y_test = test_p["fit"]
print(f"train {len(train_p)}, test {len(test_p)}")

## Fit on train, score test once

The next cell is the only place the test split is scored. It fits the three models and stores one probability table per model in `test_proba`.

In [ ]:
# The test split is scored here, once. Later sections reuse test_proba.
final = {"MajorityClass": models.MajorityClass()}
final.update({name: getattr(models, name)(**params) for name, params in models.BEST_PARAMS.items()})
print({name: params for name, params in models.BEST_PARAMS.items()})

test_proba = {name: model.fit(train_p).predict_proba(test_p) for name, model in final.items()}

## Results table

Each interval comes from 1000 bootstrap resamples of the test rows, drawn with a fixed seed. The models are already fitted, so the interval shows how much the score moves with the choice of test rows. It does not include the variation from refitting on a different training sample.

In [ ]:
N_BOOT, BOOT_SEED = 1000, 0

rows = {}
for name, proba in test_proba.items():
    ci = metrics.bootstrap_ci(y_test, proba, n_boot=N_BOOT, seed=BOOT_SEED)
    rows[name] = {
        "average AUC": metrics.average_auc(y_test, proba),
        "AUC 95% interval": "[{:.4f}, {:.4f}]".format(*ci["average AUC"]),
        "macro-F1": metrics.macro_f1(y_test, proba.idxmax(axis=1)),
        "macro-F1 95% interval": "[{:.4f}, {:.4f}]".format(*ci["macro-F1"]),
    }
test_results = pd.DataFrame(rows).T
test_results = test_results.astype({"average AUC": float, "macro-F1": float}).round(4)
print(test_results.to_string())
test_results

## Does the main model win?

Yes, on both metrics. On the full data the latent factor model scores an average AUC of 0.7585 (95% interval 0.7510 to 0.7651) against 0.6203 for logistic regression and 0.5000 for the majority baseline. Its macro-F1 is 0.5295 (0.5209 to 0.5373) against 0.3316 and 0.2830. The gap to the course model is 0.138 in AUC and 0.198 in macro-F1. Each interval is about 0.013 to 0.016 wide, so the gaps are roughly ten times the uncertainty and no interval overlaps another. Test scores sit close to the validation scores from `03_models.ipynb` (0.7576 and 0.5231 for the main model, 0.6188 and 0.3341 for logistic regression), so tuning on validation did not overfit it.

The interval covers the choice of test rows only. It does not cover refitting on a different training sample. The committed copy of this notebook runs on the 200-row sample in CI, so its table differs from the numbers quoted here, which come from the full data.

## Why average AUC and macro-F1

About 74% of the labels are "fit", so accuracy rewards the wrong behavior. Always answering "fit" scores 0.7377 accuracy on the test split, which beats both trained models (0.6169 for the main model and 0.3521 for logistic regression, whose balanced class weights push predictions toward small and large). Accuracy would rank the model that learned nothing first.

**Average AUC** (primary) ignores the class mix and any cutoff. For each class it asks whether rows of that class get higher scores than all other rows, then averages the three answers with equal weight. A rare class such as "small" counts as much as "fit", and a constant score earns exactly 0.5. **Macro-F1** (secondary) looks at the decision itself. It scores the hard labels class by class and averages with equal weight, so a model that never predicts small or large loses two thirds of the score.

Each metric covers a blind spot of the other. AUC says nothing about which cutoff to use or whether the probabilities are calibrated. Macro-F1 depends on the argmax rule and says nothing about the ranking behind it. The table below adds accuracy for contrast only.

In [ ]:
accuracy = pd.Series(
    {name: float((proba.idxmax(axis=1) == y_test).mean()) for name, proba in test_proba.items()},
    name="accuracy",
)
print(f"share of 'fit' among test labels: {(y_test == 'fit').mean():.3f}")
metric_view = pd.concat([test_results[["average AUC", "macro-F1"]], accuracy], axis=1).round(4)
print(metric_view.to_string())
metric_view

## Plot 1: ROC curves for the main model

One curve per class, each class scored against the other two. Average AUC is the mean of the three areas.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

MAIN = "LatentFactorFit"
COLORS = {"small": "#2a78d6", "fit": "#eb6834", "large": "#1baf7a"}  # reference palette, slots 1 to 3
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e1"


def style(ax):
    ax.grid(color=GRID, lw=1)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c9c8c3")
    ax.tick_params(colors=MUTED)


fig, ax = plt.subplots(figsize=(5.5, 5))
class_auc = {}
for k in metrics.CLASSES:
    is_k = (y_test == k).to_numpy()
    fpr, tpr, _ = roc_curve(is_k, test_proba[MAIN][k])
    class_auc[k] = roc_auc_score(is_k, test_proba[MAIN][k])
    ax.plot(fpr, tpr, color=COLORS[k], lw=2, label=f"{k} vs the rest, AUC {class_auc[k]:.3f}")
ax.plot([0, 1], [0, 1], color="#c9c8c3", lw=1)
assert abs(np.mean(list(class_auc.values())) - metrics.average_auc(y_test, test_proba[MAIN])) < 1e-9
style(ax)
ax.set_xlabel("false positive rate", color=MUTED)
ax.set_ylabel("true positive rate", color=MUTED)
ax.set_title(f"Test ROC curves, {MAIN}", color=INK)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

**Takeaway.** Small and large are ranked well (AUC 0.789 and 0.800), and "fit" is the weakest class at 0.686. The middle class is the one that both extremes spill into, which pulls the average down to 0.7585.

## Plot 2: where the model works

Test rows are grouped by how much the training split says about their item, and about their user. The first panel counts how many training rows share the item, the second how many share the user. Each bar is the average AUC on that group, and the course model is shown beside the main model. A group that lacks one of the three classes has no AUC and is left empty.

In [ ]:
def group_auc(mask, name):
    sub = y_test[mask]
    if sub.nunique() < len(metrics.CLASSES):
        return np.nan
    return metrics.average_auc(sub, test_proba[name][mask])


item_rows = test_p["item_id"].map(train_p["item_id"].value_counts()).fillna(0)
user_rows = test_p["user_id"].map(train_p["user_id"].value_counts()).fillna(0)
groupings = {
    "training rows for the item": pd.cut(
        item_rows, [-1, 0, 4, 19, 99, np.inf], labels=["0", "1 to 4", "5 to 19", "20 to 99", "100+"]
    ),
    "training rows for the user": pd.cut(user_rows, [-1, 0, 1, np.inf], labels=["0", "1", "2+"]),
}
by_group = {
    title: pd.DataFrame(
        {
            "rows": g.value_counts().sort_index(),
            **{
                name: [group_auc((g == lab).to_numpy(), name) for lab in g.cat.categories]
                for name in ("LogisticRegression", MAIN)
            },
        }
    )
    for title, g in groupings.items()
}
for title, table in by_group.items():
    print(title)
    print(table.round(4).to_string(), "\n")

SERIES = {"LogisticRegression": "#eb6834", MAIN: "#2a78d6"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True, gridspec_kw={"width_ratios": [5, 3]})
for ax, (title, table) in zip(axes, by_group.items(), strict=True):
    x = np.arange(len(table))
    for j, (name, color) in enumerate(SERIES.items()):
        ax.bar(x + (j - 0.5) * 0.36, table[name].fillna(0), width=0.34, color=color, label=name)
        if name == MAIN:
            for xi, v in zip(x, table[name], strict=True):
                if not np.isnan(v):
                    ax.text(xi + 0.18, v + 0.008, f"{v:.2f}", ha="center", color=INK, fontsize=9)
    ax.axhline(0.5, color="#c9c8c3", lw=1)
    ax.set_xticks(
        x, [f"{lab}\nn={n:,}" for lab, n in zip(table.index, table["rows"], strict=True)], color=MUTED
    )
    ax.set_xlabel(title, color=MUTED)
    style(ax)
axes[0].set_ylabel("average AUC on the group", color=MUTED)
axes[0].set_ylim(0.4, 0.95)
axes[0].legend(frameon=False, loc="upper left")
fig.suptitle("Test average AUC by how much the training split knows", color=INK)
fig.tight_layout()
plt.show()

**Takeaway.** Knowing the item matters most: the main model scores 0.62 on the 40 rows with an unseen item and 0.71 with one to four training rows, then 0.76 to 0.77 from five rows up (0.74 for the most rented items, a dip this notebook does not explain). Knowing the user matters little: 0.75 for unseen users against 0.77 for users with two or more training rows. The course model stays between 0.56 and 0.63 in every group.

## Plot 3: what the main model predicts

Each row is the true class. Each cell is the share of that class the model labels with the column class, so every row sums to 100%.

In [ ]:
from sklearn.metrics import confusion_matrix

predicted = test_proba[MAIN].idxmax(axis=1)
counts = confusion_matrix(y_test, predicted, labels=list(metrics.CLASSES))
share = counts / counts.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4.4))
ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{share[i, j]:.0%}", ha="center", va="center", color=INK, fontsize=12)
ax.set_xticks(range(3), metrics.CLASSES, color=MUTED)
ax.set_yticks(
    range(3), [f"{c}\nn={n:,}" for c, n in zip(metrics.CLASSES, counts.sum(axis=1), strict=True)], color=MUTED
)
ax.set_xlabel("predicted", color=MUTED)
ax.set_ylabel("true", color=MUTED)
ax.set_title(f"Test confusion matrix, {MAIN}", color=INK)
for side in ax.spines.values():
    side.set_visible(False)
fig.tight_layout()
plt.show()

**Takeaway.** The model recovers about 60% of each class. Small and large are rarely swapped (6% of small rows are labelled large and 5% of large rows small). The errors run through "fit": 35% of small rows and 36% of large rows are labelled "fit", and 19% and 18% of fit rows are labelled small and large.

In [ ]:
once = (train_p["user_id"].value_counts() == 1).mean()
print(f"training users with exactly one row: {once:.3f} of {train_p['user_id'].nunique():,}")
print(f"test rows whose user appears in train: {(user_rows > 0).mean():.3f}")
print(f"test rows whose item appears in train: {(item_rows > 0).mean():.3f}")
print(f"test rows with a user seen exactly once in train: {(user_rows == 1).mean():.3f}")

## Negative and surprising results

- **Latent vectors ($K > 0$) did not help.** On validation, average AUC falls from 0.7576 at $K = 0$ to 0.7074, 0.7165 and 0.7161 at $K = 2, 4, 8$, and none of those three fits converged within 2000 iterations. At 200 iterations $K = 4$ scored 0.7439, so more training made it worse (`03_models.ipynb`). A user vector can only be learned from that user's own rows, and 70.8% of training users appear once. Only 58.5% of test rows have a user seen in training, and 19.0% have a user seen exactly once. The item bias already carries most of what an item says about fit (item biases alone reach 0.750 on validation), so the extra item vectors mostly add parameters to fit noise. This reading fits the numbers above but was not tested directly.
- **User terms are weak.** User biases alone score 0.631 on validation against 0.750 for item biases alone. On test, rows with an unseen user score 0.754 and rows with two or more training rows for the user score 0.772, a gain of about 0.018 from the user's history.
- **Logistic regression barely moves with tuning.** Its eight validation settings span AUC 0.6171 to 0.6188, and its test AUC is 0.6203. Size and body measurements say little about whether a particular dress fits, since the same measurements rent many dresses that run differently. Only the item identity carries that signal, and this model has no access to it.
- **"Fit" is the hardest class** (ROC AUC 0.686 against 0.789 and 0.800 for the other two), and the most rented items score lower (0.74) than items with 20 to 99 training rows (0.77). I did not investigate the second pattern.

## Comparison with reported results

The two papers that report numbers on Rent the Runway are Misra, Wan and McAuley (2018) and Sheikh et al. (2019). Their numbers are copied from `docs/related_work.md`, which read them from the published tables. Our numbers come from the test predictions above. The table lines up each reported model with the closest model of ours. Read it as context, not as a ranking: the caveats below apply to every row.

In [ ]:
from sklearn.metrics import roc_auc_score

# Micro AUC pools the three one-vs-rest problems. Computed for every model to show why it is not compared.
onehot = pd.get_dummies(y_test)[list(metrics.CLASSES)]
micro = {
    name: roc_auc_score(onehot, proba[list(metrics.CLASSES)], average="micro")
    for name, proba in test_proba.items()
}
print("micro AUC on the test split:", {name: round(v, 3) for name, v in micro.items()})

reported = [
    # (source, reported model, reported metric, reported value, our model)
    ("Misra et al. 2018", "1-LV-LR", "average AUC", 0.61, "LogisticRegression"),
    ("Misra et al. 2018", "K-LF-ML (proposed)", "average AUC", 0.719, MAIN),
]
comparison = pd.DataFrame(
    [
        {
            "source": src,
            "reported model": model,
            "reported metric": metric,
            "reported": value,
            "our model": ours_name,
            "our metric": "average AUC",
            "ours": metrics.average_auc(y_test, test_proba[ours_name]),
        }
        for src, model, metric, value, ours_name in reported
    ]
)
comparison["ours minus reported"] = comparison["ours"] - comparison["reported"]
print(comparison.round(3).to_string(index=False))
comparison.round(3)

**Where we match and where we differ (full-data numbers).** Our course model scores an average AUC of 0.620, next to the 0.61 that Misra et al. report for their logistic regression baseline. Our main model scores 0.758, above their proposed model (0.719). The simple baseline lands close to the published one, and our main model lands above the published proposed model on the same metric.

**Why Sheikh et al. have no row.** They report micro-averaged AUC (0.692 for their MLP baseline, 0.749 for SFnet). Micro averaging pools the three one-vs-rest problems into one ranking, and a constant score that only reflects the class frequencies already ranks "fit" rows above the rest. The majority baseline, which knows nothing about any row, scores a micro AUC of 0.804 on our test split (printed above), higher than both reported values. A micro AUC therefore says more about the label mix than about the model, and we do not compare against it.

Why the remaining gaps do not prove we beat the paper:

- **Different split.** Ours, Misra et al. and Sheikh et al. all use a random 80/10/10 split, but the split of Misra et al. is not available. Our test interval (about 0.014 wide) covers test-row sampling in one split, not the spread across different splits.
- **Different averaging.** Our average AUC is the one-vs-rest mean over three classes. Misra et al. do not say how they average.
- **Different rows.** Our loader drops 189 exact duplicate lines (192,355 rows remain). We do not know whether the paper kept them, and a duplicate that lands in both train and test would raise a score.
- **Different models and tuning.** We have no metric learning and no neural network, and we tuned only on our validation split.